# 3_2 Gold — Sales Details

Builds the **SALES_DETAILS** Gold table per spec Section 3:

- Latest qualifying sale per LEAD_ID by ACTIVITY_AT
- Qualifying activity types: `7) New Sale` and `8) New Sale [Custom Payment Plan]`
- Includes: setter/closer identity (enriched from CRM users), contract value, date of sale, program
- Excludes: cash collected
- Dedup: `ACTIVITY_ID` latest by `DATE_UPDATED DESC → INSERT_DATE DESC`, then latest sale per `LEAD_ID` by `ACTIVITY_AT`

In [0]:
%sql
CREATE OR REPLACE TABLE crm_ingestion.gold.sales_details AS (
    WITH deduped_sales AS (
        SELECT
            id AS activity_id,
            lead_id,
            activity_at AS date_of_sale,
            custom_activity_type_id,
            created_by,
            custom_fields,
            ROW_NUMBER() OVER(PARTITION BY id ORDER BY date_updated DESC, bronze_insert_date DESC) AS dedup_rn
        FROM
            crm_ingestion.silver.lead_activites_processed
        WHERE
            custom_activity_type_id IN (
                'actitype_3E85vFq3a06LlEzXT2N1kS',  -- 7) New Sale
                'actitype_0FNk72Q8eSYX2MVd4A2UFx'   -- 8) New Sale [Custom Payment Plan]
            )
    ),
    latest_sale AS (
        SELECT
            activity_id,
            lead_id,
            date_of_sale,
            custom_activity_type_id,
            created_by,
            custom_fields,
            ROW_NUMBER() OVER(PARTITION BY lead_id ORDER BY date_of_sale DESC) AS sale_rn
        FROM deduped_sales
        WHERE dedup_rn = 1
    )
    SELECT
        s.lead_id,
        s.activity_id,
        s.date_of_sale,
        s.custom_activity_type_id AS sale_type_id,
        s.custom_fields['cf_v385AJ8HSgepKQ3rvqo4yOA3nn49eGqz39DOqojJG5M'] AS setter_id,
        COALESCE(setter_u.full_name, s.custom_fields['cf_v385AJ8HSgepKQ3rvqo4yOA3nn49eGqz39DOqojJG5M']) AS setter_name,
        s.custom_fields['cf_Lv5lSqLOZwLrNhe5M7kWx2mF8Ge2Z23aw5NUNhbXvVS'] AS closer_id,
        COALESCE(closer_u.full_name, s.custom_fields['cf_Lv5lSqLOZwLrNhe5M7kWx2mF8Ge2Z23aw5NUNhbXvVS']) AS closer_name,
        s.custom_fields['cf_vIanPjPEit6ssajmWkcprF2V1nO1itfes8hOSnjmhfT'] AS contract_value,
        s.custom_fields['cf_mXqKxcmjnlEW223wM4lgb04XyYwDm0GJ9v99xWHPWO2'] AS program,
        s.created_by,
        COALESCE(created_u.full_name, s.created_by) AS created_by_name
    FROM latest_sale s
    LEFT JOIN crm_ingestion.gold.close_crm_users_processed setter_u
        ON s.custom_fields['cf_v385AJ8HSgepKQ3rvqo4yOA3nn49eGqz39DOqojJG5M'] = setter_u.user_key
    LEFT JOIN crm_ingestion.gold.close_crm_users_processed closer_u
        ON s.custom_fields['cf_Lv5lSqLOZwLrNhe5M7kWx2mF8Ge2Z23aw5NUNhbXvVS'] = closer_u.user_key
    LEFT JOIN crm_ingestion.gold.close_crm_users_processed created_u
        ON s.created_by = created_u.user_key
    WHERE s.sale_rn = 1
)

In [0]:
%sql
SELECT *
FROM crm_ingestion.gold.sales_details
ORDER BY date_of_sale DESC
LIMIT 100